# Chat Completions API

Chat Completions API는 `messages` 배열을 입력받아 다음 대화 응답을 만드는 인터페이스이다. 각 요청에는 역할과 대화 순서를 함께 보내므로, 애플리케이션이 이전 대화를 직접 누적해야 한다.

이 노트북은 역할 기반 메시지, 대화 누적, 스트리밍, 사용량과 토큰 비용 추정을 다룬다. 새 텍스트 앱에서는 Responses API가 기본 권장 경로이지만, 기존 `messages` 기반 서비스와 API 구조를 이해하려면 Chat Completions도 알아야 한다.


## 메시지와 주요 요청 값

Chat Completions API는 대화 이력인 `messages`를 입력으로 받는다. 결과는 하나의 assistant 메시지로 반환한다. 서버가 호출 사이의 대화를 자동으로 기억하지 않으므로, 애플리케이션이 이전 메시지를 순서대로 다시 보내야 한다.

메시지 역할은 다음과 같다.

- `system`: 모델의 역할, 태도, 응답 제약을 지정한다.
- `user`: 사용자의 질문이나 요청을 담는다.
- `assistant`: 이전 모델 응답을 담아 다음 요청의 문맥으로 사용한다.

주요 요청 값은 다음과 같다.

- `model`: 호출할 모델 ID이다. 저비용 예시는 `gpt-5.6-luna`를 사용한다.
- `messages`: 역할과 내용으로 구성한 대화 이력 배열이다.
- `stream`: `True`이면 완성 문장 대신 부분 응답 청크를 순서대로 받는다.
- 생성 제어 값은 모델별 지원 범위를 확인한다. GPT-5.6 계열은 Responses API의 `reasoning.effort`를 먼저 비교한다.

공식 문서는 다음과 같다.

- [Chat Completions 생성 API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [텍스트 생성 가이드](https://developers.openai.com/api/docs/guides/text)
- [GPT-5.6 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)


### API 클라이언트 준비

이미 학습한 `.env` 설정을 불러와 `OPENAI_API_KEY`를 현재 커널에 등록한다. 키 값은 출력하지 않으며, `OpenAI()`가 환경 변수에서 키를 읽어 이후 요청에 사용한다.


In [1]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든 뒤 다시 실행한다.")
load_dotenv(dotenv_path, override=False)

client = OpenAI()

print("OpenAI 클라이언트 준비 완료")


OpenAI 클라이언트 준비 완료


## 역할 기반 메시지와 단일 응답

첫 요청은 `system`, `user`, `assistant`, `user` 순서의 대화 이력을 모델에 보낸다.

- `response.choices[0].message.content`는 이번 호출의 완성 응답이다.
- 반복 대화에서는 이 값을 다음 `messages`에 assistant 역할로 추가한다.


### 이름을 문맥에서 다시 찾기

입력 `messages`에는 사용자가 이름을 말한 턴과 이전 assistant 응답이 함께 들어간다. 모델은 그 이력을 변환해 이번 질문의 응답을 `response`에 만들고, 출력 문자열은 다음 턴의 문맥으로 저장할 수 있다.


In [3]:
MODEL_NAME = "gpt-5.6-luna"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        # 공통 지침
        {"role":"system", "content":"너는 친절한 챗봇이다."},

        # 이전 대화 내용
        {"role":"user", "content":"안녕, 내 이름은 홍길동이야."},
        {"role":"assistant", "content":"안녕, 홍길동. 무엇을 도와줄까?"},

        # 새로운 대화 내용
        {"role":"user", "content":"잘 지냈어? 내 이름을 기억하니?"},
    ] # 모델이 이전 대화를 기억할 수 있게하는 리스트
)

assistant_message = response.choices[0].message.content
print(assistant_message)

# 토큰 사용량 출력 함수
def used_tokens(response):
    print("prompt_tokens:", response.usage.prompt_tokens)
    print("completion_tokens:", response.usage.completion_tokens)
    print("total_tokens:", response.usage.total_tokens)

used_tokens(response)

잘 지냈어! 네 이름은 **홍길동**이야. 🙂
prompt_tokens: 69
completion_tokens: 20
total_tokens: 89


### 긴 대화 이력으로 요약 요청 만들기

이 예시는 Transformer 설명, 쉬운 비유 요청, 마지막 요약 요청을 하나의 `messages` 배열에 누적한다. 마지막 응답은 앞선 대화 전체를 입력으로 사용하므로, 단일 프롬프트보다 긴 문맥을 반영하는 결과를 확인할 수 있다.


In [4]:
# chat completion을 이용해서 응답을 받은 후
# - 응답 결과, 토큰 사용량 출력

messages=[
        {"role": "system", "content": "너는 LLM 전문가이다."},
        {"role": "user", "content": "안녕, 나는 LLM을 배우는 차은우야."},
        {"role": "assistant", "content": "안녕, 차은우. LLM에서 궁금한 점을 말해 줘."},
        {"role": "user", "content": "Transformer 모델을 공부하고 싶어."},
        {"role": "assistant", "content": "Transformer는 인코더와 디코더, 자기 주의, 위치 정보로 문맥을 처리하는 구조이다. 번역과 요약에 사용할 수 있다."},
        {"role": "user", "content": "어려워. 어텐션을 초등학생도 이해할 수 있게 설명해 줘."},
        {"role": "assistant", "content": "어텐션은 문장에서 중요한 단어에 더 집중하도록 가중치를 주는 방법이다. 예를 들어 강아지 이야기에서는 강아지와 관련된 단어를 더 참고한다."},
        {"role": "user", "content": "이해가 되는 것 같아. Transformer 내용을 Markdown 문서로 요약해 줘."},
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages
)

print(response.choices[0].message.content)
used_tokens(response)

# Transformer 요약

## 1. Transformer란?

**Transformer**는 문장 속 단어들의 관계를 파악해 텍스트를 이해하고 생성하는 딥러닝 모델이다.

2017년 논문 *Attention Is All You Need*에서 소개되었으며, 현재의 많은 LLM의 기본 구조로 사용된다.

예시:

- 번역
- 요약
- 질의응답
- 문장 생성
- 감정 분석
- 코드 생성

---

## 2. 핵심 아이디어: 어텐션

### 어텐션이란?

문장 안에서 **현재 단어를 이해할 때 어떤 단어를 더 중요하게 참고할지 결정하는 방법**이다.

예를 들어 다음 문장이 있다.

> 고양이가 소파 위에서 잠을 잔다.

`잠을 잔다`라는 표현을 이해할 때 모델은 `고양이`와 `소파` 같은 단어를 참고한다.  
이때 더 관련 있는 단어에 높은 중요도를 부여한다.

이를 사람에 비유하면, 문장을 읽을 때 중요한 부분에 시선을 집중하는 것과 비슷하다.

---

## 3. Self-Attention

**Self-Attention**은 한 문장 안의 단어들이 서로를 참고하는 방식이다.

각 단어는 다음 세 가지 정보로 변환된다.

- **Query**: 내가 어떤 정보를 찾고 있는가?
- **Key**: 내가 어떤 정보를 가지고 있는가?
- **Value**: 실제로 전달할 정보는 무엇인가?

어텐션은 Query와 Key를 비교해 관련성을 계산하고, 그 결과를 이용해 Value를 가중합한다.

### 기본 수식

```text
Attention(Q, K, V)
= softmax(QKᵀ / √dₖ)V
```

- `Q`: Query
- `K`: Key
- `V`: Value
- `dₖ`: Key 벡터의 차원
- `softmax`: 중요도를 확률처럼 변환하는 함수

---

## 4. Multi-Head Attention

Transformer는 어텐션을 한 번만 사용하지 않고, 여러 개의 어텐션을 동시에 사용한다.

이를 **Multi-Head Attention**

## 대화 이력 누적

반복 대화에서는 사용자 입력을 `user` 메시지로 넣고, 완성된 모델 응답을 `assistant` 메시지로 다시 넣는다. 두 값을 모두 누적해야 다음 질문에서 모델이 직전 질문과 자신의 답변을 함께 읽는다.


### 완성 응답을 messages에 저장하는 챗봇

`input()`이 만든 사용자 문자열을 `messages`에 추가한 뒤 Chat Completions를 호출한다. 반환된 `assistant_message`를 같은 배열에 추가하므로, 다음 반복의 입력이 이전 대화 전체가 된다.


In [6]:
messages = [{"role":"system", "content":"너는 까칠한 챗봇이야."}]

while True:
    user_input = input("사용자 입력 (종료는 exit)")

    if user_input.lower().strip() == 'exit':
        print("종료합니다")
        break

    # 사용자 입력 내용을 messages에 누적
    messages.append({"role":"user", "content":user_input})

    # openai chat completions api로 요청
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )

    # api 응답(==llm 응답)을 messages에 누적
    assistant_message = response.choices[0].message.content
    messages.append({"role":"assistant", "content":assistant_message})
    print("Assistant:", assistant_message)

used_tokens(response)

Assistant: 그래, 홍길동. 무슨 일이야?
Assistant: 오늘은 **떡만두국** 먹어. 든든하고 실패 확률도 낮아.  
더운 날씨에 입맛 없거나 깔끔하고 시원한 게 당기면 냉면이고—그게 아니라면 괜히 고민하지 말고 떡만두국 가.
Assistant: 그럼 **일단 오늘 할 일 중 꼭 끝내야 하는 것만 추려서 마무리하고 가**. 전부 완벽하게 하려다 집도 못 가고 기분만 망친다.

- **퇴근/하교 가능**: 지금 바로 정리하고 가. 집에 가고 싶은데 버틸 이유가 딱히 없으면 버틸 필요도 없어.
- **당장 못 감**: 10분만 자리 비우고 물 마신 뒤, 가장 급한 일 하나만 처리해.
- **너무 지쳤거나 불안함**: “오늘은 여기까지 하겠습니다”라고 말하고 가는 것도 방법이야. 무단으로 사라지지만 않으면 된다.

결론: **오늘 꼭 해야 하는 일 1개만 끝내고 집에 가.**
종료합니다
prompt_tokens: 163
completion_tokens: 230
total_tokens: 393


## 스트리밍 응답과 청크 누적

스트리밍은 한 번에 완성된 메시지를 받지 않고 `delta.content` 조각을 순서대로 받는다. 화면 출력만 하면 다음 턴에 넣을 완성 응답이 없으므로, 각 조각을 리스트에 모아 `"".join()`으로 합쳐야 한다.


### 단일 요청의 스트리밍 청크 출력

`stream=True`는 응답을 청크 반복자로 바꾼다. 각 청크의 텍스트를 즉시 출력해 생성 과정을 보며, 같은 텍스트를 누적하면 파일 저장이나 대화 이력 추가에도 사용할 수 있다.


In [7]:
stream = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role": "user", "content": "스트리밍을 설명하는 30줄 짜리 응답을 만들어 줘."}],
    stream=True,
)

# 스트리밍은 비어 있지 않은 새 텍스트만 누적해 전체 응답을 만든다.
response_parts = []
for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        response_parts.append(content)
        print(content, end="", flush=True)

assistant_message = "".join(response_parts)
print()


1. 스트리밍은 인터넷을 통해 콘텐츠를 조금씩 전송하고 재생하는 기술입니다.  
2. 사용자는 파일 전체가 다운로드될 때까지 기다리지 않아도 됩니다.  
3. 데이터가 도착하는 즉시 음악, 영상, 게임 등을 이용할 수 있습니다.  
4. 대표적인 스트리밍 서비스로는 유튜브, 넷플릭스, 스포티파이가 있습니다.  
5. 동영상 스트리밍은 영상을 작은 데이터 조각으로 나누어 전송합니다.  
6. 재생 기기는 받은 조각을 순서대로 화면에 표시합니다.  
7. 동시에 다음 장면에 필요한 데이터도 계속 받아옵니다.  
8. 이를 위해 기기에는 일정량의 데이터가 임시로 저장됩니다.  
9. 이 임시 저장 공간을 버퍼라고 합니다.  
10. 네트워크가 불안정하면 버퍼가 부족해 재생이 멈출 수 있습니다.  
11. 반대로 연결 상태가 좋으면 고화질 콘텐츠를 원활하게 볼 수 있습니다.  
12. 스트리밍 품질은 인터넷 속도와 지연 시간의 영향을 받습니다.  
13. 사용 중인 기기의 성능과 화면 해상도도 중요한 요소입니다.  
14. 많은 서비스는 네트워크 상황에 따라 화질을 자동으로 조절합니다.  
15. 이를 적응형 비트레이트 스트리밍이라고 합니다.  
16. 비트레이트는 일정 시간 동안 전송되는 데이터의 양을 의미합니다.  
17. 비트레이트가 높을수록 일반적으로 화질과 음질이 좋아집니다.  
18. 하지만 더 빠른 인터넷 연결과 많은 데이터 사용량이 필요합니다.  
19. 스트리밍은 실시간 스트리밍과 주문형 스트리밍으로 나눌 수 있습니다.  
20. 실시간 스트리밍은 방송이나 경기처럼 현재 진행되는 콘텐츠를 전달합니다.  
21. 주문형 스트리밍은 사용자가 원하는 시간에 콘텐츠를 재생하는 방식입니다.  
22. 온라인 회의와 게임 방송도 실시간 스트리밍의 사례입니다.  
23. 스트리밍에는 콘텐츠를 제공하는 서버와 이용자의 기기가 필요합니다.  
24. 콘텐츠 전송 네트워크는 사용자와 가까운 서버에서 데이터를 전달합니다.  
25. 이 구조는 전송 거리를 줄여 지연과 끊

In [9]:
print(len(response_parts))
print(response_parts[0:10])

626
['1', '.', ' 스트', '리', '밍', '은', ' 인터넷', '을', ' 통해', ' 콘텐츠']


### 스트리밍 챗봇의 완성 응답 저장

반복 챗봇에서도 청크를 `response_parts`에 모아 이번 턴의 `assistant_message`를 만든다. 이렇게 갱신한 문자열을 `messages`에 추가하면, 다음 질문이 스트리밍으로 생성된 직전 답변까지 참조한다.


In [10]:
# 기본 지침 설정
messages = [{"role":"system", "content":"너는 젊은 꼰대 챗봇이야."}]

while True:
    user_input = input("사용자 입력 (종료는 exit)")

    if user_input.lower().strip() == 'exit':
        print("종료합니다")
        break

    # 사용자 입력 내용을 messages에 누적
    messages.append({"role":"user", "content":user_input})

    # stream 객체 생성
    stream = client.chat.completions.create(
        model = MODEL_NAME,
        messages = messages,
        stream=True,
    )

    # 입력 청크를 누적할 리스트
    response_parts = []
    print("Assistant:", end="", flush=True)
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            response_parts.append(content)
            print(content, end="", flush=True)
    print()

    # Assistant의 응답을 다음 messages에 누적
    assistant_message = "".join(response_parts)
    messages.append({"role":"assistant", "content":assistant_message})

Assistant:안녕하세요. 저는 **젊은 꼰대 챗봇**입니다.

요즘 말투와 트렌드는 어느 정도 따라가지만, 기본적으로는  
“그건 예의가 아니지 않니?”, “검색하면 나오는 건 직접 좀 찾아보자” 같은 말을 가끔 하는 타입입니다.

그래도 잔소리만 하지는 않고, 질문에 맞춰 정보를 정리하고 아이디어를 내고 글도 다듬어 드립니다.  
다만 근거 없는 말이나 무례한 표현은 조금 지적할 수 있어요. 역시 기본이 중요하니까요.

편하게 물어보세요. 제가 아는 선에서 최대한 정확하고 쓸모 있게 답해 드리겠습니다.
종료합니다


## 토큰 수, usage와 비용 추정

토큰은 모델이 처리하는 텍스트 단위이며, 요청과 응답 토큰 수가 비용에 영향을 준다. 실제 청구에는 응답의 `usage`를 우선 사용하고, 요청 전 예상 비용은 토크나이저로 근사한다.

공식 문서: [텍스트 생성](https://developers.openai.com/api/docs/guides/text), [최신 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)


### tiktoken 설치

`tiktoken`은 문자열을 토큰 ID로 변환해 요청 전 길이를 추정하는 라이브러리이다. 설치가 끝나면 다음 셀에서 `o200k_base` 인코딩으로 같은 문장의 토큰 수를 계산한다.


In [11]:
!pip install tiktoken


  Using cached regex-2026.7.19-cp312-cp312-win_amd64.whl.metadata (41 kB)
   ---------------------------------------- 0.0/874.7 kB ? eta -:--:--
   ---------------------------------------- 874.7/874.7 kB 13.0 MB/s  0:00:00
Using cached regex-2026.7.19-cp312-cp312-win_amd64.whl (277 kB)

   ---------------------------------------- 0/2 [regex]
   -------------------- ------------------- 1/2 [tiktoken]
   -------------------- ------------------- 1/2 [tiktoken]
   -------------------- ------------------- 1/2 [tiktoken]
   ---------------------------------------- 2/2 [tiktoken]



### 최신 예시 모델의 토크나이저 선택

`o200k_base`는 GPT-5 계열 예시의 토큰 수를 근사하는 인코딩이다. 모델별 실제 토큰화와 API 사용량은 바뀔 수 있으므로, 청구나 정산에는 응답의 `usage`를 사용한다.


In [12]:
import tiktoken

gpt56_encoding = tiktoken.get_encoding("o200k_base")
print(gpt56_encoding)


<Encoding 'o200k_base'>


### 짧은 문장의 토큰 수 계산

문장을 `encode()`에 넣으면 정수 토큰 ID 목록이 나온다. 목록 길이를 세어 입력 길이를 수치로 확인하고, 다음 긴 본문과 비용 계산의 기준으로 사용한다.


In [13]:
sample_text = "아버지가 방에 들어가신다."
encoded_sample = gpt56_encoding.encode(sample_text)

print(len(encoded_sample))


10


## 긴 입력의 토큰 수와 응답 usage

긴 기사 본문을 토큰화해 요청 전 입력 길이를 추정한다. 실제 API 호출 뒤에는 `response.usage`의 입력·출력 토큰 수를 읽어 추정값보다 신뢰도 높은 비용 근거로 사용한다.


### 기사 본문의 입력 토큰 수 추정

긴 문자열 `text`를 준비하고 `gpt56_encoding`으로 토큰 ID 목록을 만든다. 출력된 길이는 다음 요약 요청의 예상 입력량과 비용 계산 함수의 입력이 된다.


In [14]:
text = """
정부가 인공지능(AI) 3대 강국으로 도약하기 위한 핵심 인프라 구축을 본격 시작했다. 정부는 전남광주 해남군에 AI 반도체 1만5000개를 갖춘 국가 AI 컴퓨팅센터를 기반으로 국내 AI 연구개발과 서비스 개발에 필요한 컴퓨팅 자원을 공급한다는 계획이다. 이와 함께 AI 인재 양성 프로젝트도 함께 가동해 ‘AI 풀스택 국가’ 구축에 속도를 낸다는 방침이다.

과학기술정보통신부는 3일 해남군 솔라시도 데이터센터 파크에서 국가 AI 컴퓨팅센터 착공식을 열었다고 밝혔다. 국가 AI 컴퓨팅센터는 정부가 추진하는 AI 고속도로의 핵심 인프라다.

삼성SDS 컨소시엄이 구축을 맡았으며 정부와 국민성장펀드, 민간이 공동 출자해 설립한 특수목적법인(SPC) ‘한국AI컴퓨팅센터(KOACC·코아크)’가 건설과 운영을 담당한다.
GPU 1.5만개 품는 AI 컴퓨팅센터, 해남서 첫삽이미지 크게보기
센터엔 2028년까지 그래픽처리장치(GPU) 1만5000개가 투입된다. 내년부터 삼성SDS 데이터센터에 일부 AI 자원을 먼저 개방해 산·학·연의 GPU 수요에 대응할 계획이다. 완공되면 연구개발(R&D)은 물론 국산 AI 반도체 검증과 상용화 테스트베드 역할도 맡는다. 정부는 R&D 전용 구역을 별도로 조성해 국내 AI 반도체 생태계 육성도 지원할 예정이다. 총 사업비는 약 2조5000억원이다.

배경훈 부총리 겸 과기정통부 장관은 이날 착공식에서 “국가 AI 컴퓨팅센터는 단순한 데이터센터가 아니라 토큰을 생산하는 AI 팩토리”라며 “대한민국이 메모리 반도체 공급국을 넘어 AI 공급망의 핵심 국가로 성장하는 기반이 될 것”이라고 강조했다. 그러면서 “정부도 AI 혁명의 골든타임을 놓치지 않도록 끝까지 지원하겠다”고 강조했다. 삼성SDS 출신인 안정태 코아크 대표는 “오늘의 착공이 대한민국을 AI 3대 강국으로 이끄는 출발점이 될 것”이라고 화답했다.

정부는 이날 국민 AI 활용 역량을 높이기 위한 ‘모두의 AI 성장 사다리 프로젝트’도 동시에 출범시켰다. 온라인 교육 플랫폼인 ‘모두의 AI 배움터’, AI 개발 플랫폼 ‘모두의 AI 실험실’, 지역 실증 공간인 ’모두의 AI 라운지’를 연결해 교육부터 개발, 창업까지 이어지는 체계를 구축한다는 구상이다.

특히 오프라인 거점인 모두의 AI 라운지에서는 시민이 개발한 AI 서비스를 직접 실증할 수 있다. 올해 수도권, 강원, 충청, 경상, 전라·제주 등 전국 5개 권역에 조성되며 AI 전문가가 개발을 지원한다.

배 부총리는 이날 국립광주과학관에서 열린 출범식에서 “AI는 이제 한글처럼 누구나 익히고 활용해야 하는 새로운 기본 역량”이라며 “배움터에서 배우고, 실험실에서 개발하고, 라운지에서 실증하는 선순환 구조를 마련해 AI 혜택을 모두가 누리는 AI 기본사회를 구축하겠다”고 말했다.
"""

encoded_text = gpt56_encoding.encode(text)
print("gpt-5.6-luna 예상 입력 토큰 수:", len(encoded_text))


gpt-5.6-luna 예상 입력 토큰 수: 800


### 요약 응답의 usage 읽기

기사 `text`를 user 메시지로 보내고, `response.usage`에서 실제 입력·출력 토큰 수를 읽는다. 출력 텍스트는 비용 계산에, usage 값은 추정 토큰 수 검증에 사용한다.


In [15]:
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": "제공된 뉴스 기사의 핵심을 간결하게 요약하는 챗봇이다."},
        {"role": "user", "content": text},
    ],
)

output_text = response.choices[0].message.content
usage = response.usage
print("요약 응답:", output_text)
print("입력 토큰:", usage.prompt_tokens)
print("출력 토큰:", usage.completion_tokens)


요약 응답: 정부가 AI 3대 강국 도약을 위해 전남 해남에 국가 AI 컴퓨팅센터를 착공했다. 삼성SDS 컨소시엄이 구축하는 이 센터에는 2028년까지 GPU 1만5000개가 투입되며, AI 연구개발과 국산 AI 반도체 검증·상용화를 지원한다. 총사업비는 약 2조5000억 원이다.

이와 함께 정부는 AI 교육·개발·실증을 연계한 ‘모두의 AI 성장 사다리 프로젝트’도 출범시켰다. 온라인 교육 플랫폼, 개발 플랫폼, 지역 실증 공간을 통해 국민의 AI 활용 역량과 창업을 지원하고 ‘AI 풀스택 국가’ 구축을 추진할 계획이다.
입력 토큰: 831
출력 토큰: 172


### 기준일이 있는 토큰 비용 계산

아래 값은 2026-08-03에 공식 모델 문서에서 확인한 표준 처리 단가이다.

- `gpt-5.6-luna` 입력: 1M 토큰당 `$0.20`이다.
- `gpt-5.6-luna` 캐시 입력: 1M 토큰당 `$0.02`이다.
- `gpt-5.6-luna` 출력: 1M 토큰당 `$1.20`이다.
- 입력이 272K 토큰을 넘으면 전체 요청에 입력 2배와 출력 1.5배 단가가 적용된다.
- 캐시 쓰기는 캐시되지 않은 입력 단가의 1.25배이다.

가격과 처리 등급은 바뀔 수 있다. 운영 전에는 공식 페이지를 다시 확인한다.

- [GPT-5.6 Luna 모델 문서](https://developers.openai.com/api/docs/models/gpt-5.6-luna)
- [Pricing](https://developers.openai.com/api/docs/pricing)


In [17]:
PRICING = {
    "gpt-5.6-luna": {"input": 0.20, "output": 1.20},  # USD / 1M tokens
}

def calc_cost(input_tokens, output_tokens, model="gpt-5.6-luna"):
    price = PRICING[model]
    input_cost = input_tokens / 1_000_000 * price["input"]
    output_cost = output_tokens / 1_000_000 * price["output"]
    return input_cost + output_cost

request_cost = calc_cost(usage.prompt_tokens, usage.completion_tokens)
print(f"이번 요청의 추정 비용: ${request_cost:.8f}")

print(calc_cost(1000, 2000) * 10000)

이번 요청의 추정 비용: $0.00037260
26.0


## 신규 프로젝트 권장 경로: Responses API

Chat Completions는 계속 지원되지만, OpenAI는 신규 프로젝트에 Responses API를 권장한다. Responses API는 텍스트·이미지·도구 호출과 멀티턴 상태를 하나의 인터페이스에서 처리하므로 최신 추론 모델과 에이전트 기능을 연결하기 쉽다.

- `input`에 사용자 요청을 전달한다.
- `output_text`에서 최종 텍스트를 읽는다.
- `previous_response_id`로 이전 응답을 이어 멀티턴 대화를 구성할 수 있다.
- `reasoning.effort`로 작업에 사용할 추론 수준을 조정한다.
- Chat Completions는 `messages`와 역할 기반 대화 구조를 학습하거나 기존 서비스를 유지할 때 사용한다.

공식 문서는 다음과 같다.

- [Responses 생성 API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Chat Completions에서 Responses로 이전](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [GPT-5.6 모델 지침](https://developers.openai.com/api/docs/guides/latest-model)
